# Scribble Conditioning Comparison

We compare four scribble conditioning strategies under a **neutral prompt**, measuring how well each matches a target distribution of 25% man / 75% woman portraits.

**Conditions**
- `man_scribble` — HED edge map extracted from a generated man portrait
- `woman_scribble` — HED edge map extracted from a generated woman portrait
- `avg_scribble` — pixel mean of the two HED maps
- `sdedit_scribble` — man scribble denoised toward a gender-neutral appearance via SDEdit

**Protocol (repeated over `N_RUNS` seeds)**
1. Generate source portraits → extract scribbles → run all four conditions.
2. Record MMD (CLIP space) vs a freshly drawn target distribution.
3. After all runs: for each method pick the **best scribble** (lowest MMD across seeds).
4. From that best scribble generate **2 000 images**, compute final MMD, and compute a 95 % CI for the proportion of male images (binomial, normal approximation).

## 1. Setup

In [ ]:
import sys, os

if 'google.colab' in str(get_ipython()):
    !pip install -q diffusers transformers accelerate controlnet_aux scikit-learn peft

    from huggingface_hub import login
    from google.colab import userdata
    import getpass

    hf_token = userdata.get('HF') if hasattr(userdata, 'get') else None
    login(token=hf_token) if hf_token else login()

    github_token = userdata.get('GITHUB') if hasattr(userdata, 'get') else getpass.getpass('GitHub token: ')
    repo_url  = f'https://{github_token}@github.com/orineo1/conditional-matching-paper.git'
    repo_name = 'conditional-matching-paper'
    branch    = 'compareSDvsNaive'

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        !cd {repo_name} && git pull
    !cd {repo_name} && git checkout {branch}

    for path in [f'/content/{repo_name}', f'/content/{repo_name}/SD_cond_SD_controlnet']:
        if path not in sys.path:
            sys.path.insert(0, path)

    print(f'Repo ready on branch: {branch}')

## 2. Imports

In [ ]:
import copy, json
import numpy as np
import torch
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from PIL import Image
from scipy.stats import norm
from IPython.display import display

from models        import load_models
from image_utils   import sobel_proxy, build_base_image
from clip_utils    import load_clip_model, encode_images_clip
from generation    import generate_and_store_cs
from run_dps       import compute_clip_softmax
from metrics       import compute_mmd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 3. Config

In [ ]:
N_SOURCE         = 10     # portraits per gender used to extract scribbles each run
N_TARGET         = 2000
N_COND           = 250    # images per condition per seed (quick per-run MMD proxy)
N_FINAL          = 2000   # images from the best scribble for the final evaluation
TARGET_MAN_FRAC = 0.25  # 25% man / 75% woman
MAN_FRAC   = TARGET_MAN_FRAC        # 0.25
WOMAN_FRAC = 1.0 - TARGET_MAN_FRAC  # 0.75
N_RUNS           = 10
CONTROLNET_SCALE = 0.5
SDEDIT_START     = 15
N_STEPS          = 30
SDEDIT_CFG       = 7.5
SDEDIT_PROMPT = 'a superrealistic portrait photograph of a person, mostly woman, studio lighting'

MAN_PROMPT    = 'a superrealistic portrait photograph of a man, studio lighting'
WOMAN_PROMPT  = 'a superrealistic portrait photograph of a woman, studio lighting'
NEUTRAL_PROMPT = 'a superrealistic professional photograph of'

CONDITION_NAMES = ['man_scribble', 'woman_scribble', 'avg_scribble', 'sdedit_scribble']

print('Config OK')

## 4. Load models

In [ ]:
architect, sprinter    = load_models(device)
clip_model, clip_processor = load_clip_model(device)
print('Models loaded.')

## 5. Helpers

In [ ]:
from controlnet_aux import HEDdetector
hed = HEDdetector.from_pretrained('lllyasviel/Annotators')


def generate_and_store_cs(pipe, prompt, cond_pil, num_samples, batch_size=2, cn_scale=0.5, seed=None):
    """Generate images with ControlNet + collect latents."""
    original_vae_dtype = pipe.vae.dtype
    pipe.vae.to(dtype=torch.float16)
    all_images, all_lats = [], []

    def latents_callback(p, step_index, timestep, cb_kwargs):
        if step_index == p.num_timesteps - 1:
            p._current_latents = cb_kwargs['latents'].detach().cpu().numpy()
        return cb_kwargs

    generator = torch.Generator(device=pipe.device).manual_seed(seed) if seed is not None else None
    for i in range(0, num_samples, batch_size):
        curr = min(batch_size, num_samples - i)
        result = pipe(
            prompt=[prompt] * curr,
            image=[cond_pil] * curr,
            num_inference_steps=2,
            guidance_scale=0.0,
            controlnet_conditioning_scale=cn_scale,
            callback_on_step_end=latents_callback,
            generator=generator,
        )
        all_images.extend(result.images)
        all_lats.append(pipe._current_latents.reshape(curr, -1))
        print(f'  {len(all_images)}/{num_samples}', end='\r')
    print()
    pipe.vae.to(dtype=original_vae_dtype)
    return all_images, np.vstack(all_lats)


def generate_condition(scribble_pil, prompt, n, batch_size=2, seed=None):
    sprinter.vae.to(dtype=torch.float16)
    generator = torch.Generator(device=sprinter.device).manual_seed(seed) if seed is not None else None
    images = []
    with torch.no_grad():
        for start in range(0, n, batch_size):
            bs = min(batch_size, n - start)
            result = sprinter(
                prompt=[prompt] * bs,
                image=[scribble_pil] * bs,
                num_inference_steps=2,
                guidance_scale=0.0,
                controlnet_conditioning_scale=CONTROLNET_SCALE,
                output_type='pil',
                generator=generator,
            )
            images.extend(result.images)
            print(f'  {len(images)}/{n}', end='\r')
    sprinter.vae.to(dtype=torch.float32)
    print()
    return images


def clip_embed(images):
    """Encode a list of PIL images to CLIP embeddings."""
    tensors = torch.cat([TF.to_tensor(img).unsqueeze(0) for img in images]).to(device)
    clip_model.to(device)
    with torch.no_grad():
        embs = encode_images_clip(tensors, clip_model, clip_processor)
    clip_model.to('cpu')
    return embs


def build_target(sobel_pil, seed_offset=0, n=N_TARGET):
    n_man   = int(n * TARGET_MAN_FRAC)
    n_woman = n - n_man
    with torch.no_grad():
        man_imgs,   _ = generate_and_store_cs(sprinter, MAN_PROMPT,   sobel_pil, n_man,   2, CONTROLNET_SCALE, seed=seed_offset)
        woman_imgs, _ = generate_and_store_cs(sprinter, WOMAN_PROMPT, sobel_pil, n_woman, 2, CONTROLNET_SCALE, seed=seed_offset + 500)
    return clip_embed(man_imgs + woman_imgs)


def run_sdedit(scr_man_pil, seed):
    """SDEdit: add noise to man scribble then denoise toward neutral prompt."""
    architect.vae.to(dtype=torch.float32)
    architect.scheduler.set_timesteps(N_STEPS, device=device)
    timesteps = architect.scheduler.timesteps

    with torch.no_grad():
        t = TF.to_tensor(scr_man_pil.convert('RGB')).unsqueeze(0).to(device).float()
        t = (t * 2.0) - 1.0
        latent = architect.vae.encode(t).latent_dist.mean * architect.vae.config.scaling_factor

    t_start = timesteps[SDEDIT_START]
    alpha   = architect.scheduler.alphas_cumprod.to(device)[t_start.long()].float()
    gen     = torch.Generator(device=device).manual_seed(seed)
    noise   = torch.randn(latent.shape, generator=gen, device=device, dtype=torch.float32)
    latents = ((alpha ** 0.5) * latent + ((1 - alpha) ** 0.5) * noise).half()

    with torch.no_grad():
        prompt_embeds, neg_embeds, pooled, neg_pooled = architect.encode_prompt(
            prompt=SDEDIT_PROMPT, negative_prompt='', device=device,
            do_classifier_free_guidance=True, num_images_per_prompt=1,
        )
    add_ids = torch.tensor([[512, 512, 0, 0, 512, 512]], dtype=prompt_embeds.dtype, device=device)
    added   = {'text_embeds': torch.cat([neg_pooled, pooled]), 'time_ids': add_ids.repeat(2, 1)}
    cfg_st  = torch.cat([neg_embeds, prompt_embeds])

    sched = copy.deepcopy(architect.scheduler)
    with torch.no_grad():
        for t in timesteps[SDEDIT_START:]:
            lmi  = sched.scale_model_input(torch.cat([latents] * 2), t)
            out  = architect.unet(lmi, t, encoder_hidden_states=cfg_st, added_cond_kwargs=added, return_dict=False)[0]
            nu, nc = out.chunk(2)
            latents = sched.step(nu + SDEDIT_CFG * (nc - nu), t, latents).prev_sample

    with torch.no_grad():
        dec = architect.vae.decode((latents.float() / architect.vae.config.scaling_factor).to(architect.vae.dtype)).sample
        pil = T.ToPILImage()(torch.clamp((dec.float() + 1.0) / 2.0, 0.0, 1.0).squeeze(0).cpu())
    return pil


def binomial_ci_normal(n_success, n_total, z=1.96):
    """Normal-approximation 95% CI for binomial proportion."""
    p_hat = n_success / n_total
    se    = np.sqrt(p_hat * (1 - p_hat) / n_total)
    return p_hat, p_hat - z * se, p_hat + z * se


print('Helpers ready.')

## 6. Base oval & Sobel conditioning

In [ ]:
base_image_pil, base_tensor = build_base_image(device)
with torch.no_grad():
    sobel_pil = T.ToPILImage()(sobel_proxy(base_tensor, device).squeeze(0).cpu())

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(base_image_pil); axes[0].set_title('Base oval');   axes[0].axis('off')
axes[1].imshow(sobel_pil, cmap='gray'); axes[1].set_title('Sobel cond'); axes[1].axis('off')
plt.tight_layout(); display(fig); plt.close()

## 7. Main experiment loop

For each seed we: generate source portraits → extract scribbles (man / woman / avg / sdedit) → draw a fresh target → generate `N_COND` neutral images per condition → compute MMD and save everything.

In [ ]:
import os
SAVE_DIR = '/content/scribble_runs'
os.makedirs(SAVE_DIR, exist_ok=True)

all_records = []   # one dict per (seed, condition)

# best_per_method[cond] = {'seed': int, 'mmd': float, 'scribble': PIL}
best_per_method = {c: {'seed': None, 'mmd': float('inf'), 'scribble': None} for c in CONDITION_NAMES}

for seed in range(1, N_RUNS + 1):
    print(f"\n{'='*55}")
    print(f'  SEED {seed}/{N_RUNS}')
    print(f"{'='*55}")

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

    # ── Source portraits
    with torch.no_grad():
        man_imgs, _   = generate_and_store_cs(sprinter, MAN_PROMPT,   sobel_pil, N_SOURCE, 2, CONTROLNET_SCALE, seed=seed)
        woman_imgs, _ = generate_and_store_cs(sprinter, WOMAN_PROMPT, sobel_pil, N_SOURCE, 2, CONTROLNET_SCALE, seed=seed + 1)

    src_man, src_woman = man_imgs[0], woman_imgs[0]

    # ── HED scribbles
    scr_man   = hed(src_man,   scribble=True)
    scr_woman = hed(src_woman, scribble=True)
    avg_np = (MAN_FRAC   * np.array(scr_man).astype(np.float32) +WOMAN_FRAC * np.array(scr_woman).astype(np.float32)).clip(0, 255).astype(np.uint8)
    scr_avg   = Image.fromarray(avg_np)
    scr_sdedit = run_sdedit(scr_man, seed)

    # ── Save scribbles
    seed_dir = os.path.join(SAVE_DIR, f'seed_{seed:02d}')
    os.makedirs(seed_dir, exist_ok=True)
    scr_man.save(   os.path.join(seed_dir, 'scribble_man.png'))
    scr_woman.save( os.path.join(seed_dir, 'scribble_woman.png'))
    scr_avg.save(   os.path.join(seed_dir, 'scribble_avg.png'))
    scr_sdedit.save(os.path.join(seed_dir, 'scribble_sdedit.png'))

    run_conditions = {
        'man_scribble':    scr_man,
        'woman_scribble':  scr_woman,
        'avg_scribble':    scr_avg,
        'sdedit_scribble': scr_sdedit,
    }

    # ── Fresh target distribution for this seed
    print('  Building target distribution...')
    proxy_target_clip = build_target(sobel_pil, seed_offset=seed * 1000, n=N_COND)

    # ── Evaluate each condition
    seed_results = {}
    for cond_name, scribble in run_conditions.items():
        print(f'  Generating [{cond_name}]...')
        imgs = generate_condition(scribble, NEUTRAL_PROMPT, N_COND, seed=seed)
        embs = clip_embed(imgs)
        mmd  = compute_mmd(embs, proxy_target_clip.detach()).item()

        # Gender classification
        softmax_res, _ = compute_clip_softmax(imgs, clip_model, clip_processor, MAN_PROMPT, WOMAN_PROMPT, device)
        n_male   = sum(1 for r in softmax_res if r['label'] == 'male')
        pct_male = n_male / len(softmax_res) * 100

        record = {'seed': seed, 'condition': cond_name, 'mmd': mmd, 'pct_male': pct_male}
        all_records.append(record)
        seed_results[cond_name] = mmd

        print(f'    MMD={mmd:.5f}  male={pct_male:.1f}%')

        # Track best scribble per method
        if mmd < best_per_method[cond_name]['mmd']:
            best_per_method[cond_name]['mmd']     = mmd
            best_per_method[cond_name]['seed']    = seed
            best_per_method[cond_name]['scribble'] = scribble.copy()

    # Save per-seed results
    with open(os.path.join(seed_dir, 'results.json'), 'w') as f:
        json.dump(seed_results, f, indent=2)

print('\nAll runs complete.')

## 8. Save best scribbles

The best scribble per method (lowest per-run MMD) is saved to disk.

In [ ]:
BEST_DIR = os.path.join(SAVE_DIR, 'best_scribbles')
os.makedirs(BEST_DIR, exist_ok=True)

print('Best scribble summary (by per-run MMD proxy):')
for cond, info in best_per_method.items():
    path = os.path.join(BEST_DIR, f'best_{cond}.png')
    info['scribble'].save(path)
    print(f'  {cond:<20}  seed={info["seed"]}  proxy_mmd={info["mmd"]:.5f}  saved → {path}')

# Visualise best scribbles
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, cond in zip(axes, CONDITION_NAMES):
    ax.imshow(best_per_method[cond]['scribble'], cmap='gray')
    ax.set_title(f'{cond}\nseed {best_per_method[cond]["seed"]}  MMD={best_per_method[cond]["mmd"]:.4f}', fontsize=9)
    ax.axis('off')
plt.suptitle('Best scribble per method', fontsize=12, fontweight='bold')
plt.tight_layout(); display(fig); plt.close()

## 9. Final evaluation on best scribbles

For each method's best scribble:
- Generate **2 000** images with the neutral prompt.
- Build a fresh target distribution of **2 000** images (1 000 man + 1 000 woman).
- Compute MMD in CLIP space.
- Classify images by gender (CLIP softmax) and compute a 95 % CI for *p*(male) via normal approximation.

In [ ]:
FINAL_SEED = 9999   # fixed seed for reproducible final evaluation

print(f'Building final target distribution ({N_TARGET} images)...')
final_target_clip = build_target(sobel_pil, seed_offset=FINAL_SEED)
print(f'Target CLIP embeddings: {final_target_clip.shape}')

final_results = {}

for cond in CONDITION_NAMES:
    best_scribble = best_per_method[cond]['scribble']
    print(f'\n[{cond}]  generating {N_FINAL} images...')

    imgs = generate_condition(best_scribble, NEUTRAL_PROMPT, N_FINAL, seed=FINAL_SEED)

    # CLIP embeddings + MMD
    embs = clip_embed(imgs)
    mmd  = compute_mmd(embs, final_target_clip.detach()).item()

    # Gender classification
    softmax_res, _ = compute_clip_softmax(imgs, clip_model, clip_processor, MAN_PROMPT, WOMAN_PROMPT, device)
    n_male = sum(1 for r in softmax_res if r['label'] == 'male')

    p_hat, ci_lo, ci_hi = binomial_ci_normal(n_male, N_FINAL)

    final_results[cond] = {
        'mmd':     mmd,
        'n_male':  n_male,
        'n_female': N_FINAL - n_male,
        'p_male':  p_hat,
        'ci_lo':   ci_lo,
        'ci_hi':   ci_hi,
        'best_seed': best_per_method[cond]['seed'],
    }

    print(f'  MMD={mmd:.5f}  p(male)={p_hat:.3f}  95%CI=[{ci_lo:.3f}, {ci_hi:.3f}]  ({n_male}M / {N_FINAL - n_male}F)')

print('\nFinal evaluation complete.')

## 10. Results summary

In [ ]:
import pandas as pd

# ── Per-run aggregation table
def agg(vals):
    a = np.array([v for v in vals if not np.isnan(v)])
    return f'{a.mean():.4f} ± {a.std():.4f}' if len(a) else '—'

run_rows = {}
for cond in CONDITION_NAMES:
    recs = [r for r in all_records if r['condition'] == cond]
    run_rows[cond] = {
        'MMD (mean ± std)':    agg([r['mmd']      for r in recs]),
        '% male (mean ± std)': agg([r['pct_male'] for r in recs]),
    }
print('=== Per-run summary (N_COND images/seed) ===')
display(pd.DataFrame(run_rows).T)

# ── Final evaluation table
final_rows = {}
for cond, r in final_results.items():
    final_rows[cond] = {
        'Best seed':     r['best_seed'],
        'MMD (final)':   f"{r['mmd']:.5f}",
        'n male':        r['n_male'],
        'n female':      r['n_female'],
        'p(male)':       f"{r['p_male']:.3f}",
        '95% CI':        f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]",
    }
print('\n=== Final evaluation (N_FINAL images from best scribble) ===')
display(pd.DataFrame(final_rows).T)

## 11. Visualisation

In [ ]:
# ── MMD bar chart with per-run error bars
means = [np.mean([r['mmd'] for r in all_records if r['condition'] == c]) for c in CONDITION_NAMES]
stds  = [np.std( [r['mmd'] for r in all_records if r['condition'] == c]) for c in CONDITION_NAMES]
final_mmds = [final_results[c]['mmd'] for c in CONDITION_NAMES]

x = np.arange(len(CONDITION_NAMES))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - 0.2, means,       0.35, yerr=stds, capsize=4, alpha=0.75, label=f'Per-run proxy (N={N_COND})')
ax.bar(x + 0.2, final_mmds,  0.35, alpha=0.75, label=f'Final eval (N={N_FINAL})')
ax.set_xticks(x); ax.set_xticklabels(CONDITION_NAMES, rotation=15, ha='right')
ax.set_ylabel('MMD'); ax.set_title('MMD vs target — all conditions')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); display(fig); plt.close()

# ── p(male) CI plot
p_hats = [final_results[c]['p_male'] for c in CONDITION_NAMES]
ci_lo  = [final_results[c]['ci_lo']  for c in CONDITION_NAMES]
ci_hi  = [final_results[c]['ci_hi']  for c in CONDITION_NAMES]
errors = [[p - lo for p, lo in zip(p_hats, ci_lo)],
          [hi - p for p, hi in zip(p_hats, ci_hi)]]

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(CONDITION_NAMES, p_hats, yerr=errors, fmt='o', capsize=6, linewidth=1.5)
ax.axhline(TARGET_MAN_FRAC, color='gray', linestyle='--', linewidth=0.8, label=f'target p={TARGET_MAN_FRAC}')
ax.set_ylabel('p(male)')
ax.set_title(f'Proportion male — 95 % CI (N={N_FINAL}, normal approx.)')
ax.set_ylim(0, 1); ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout(); display(fig); plt.close()

## 12. Save results & download

In [ ]:
import shutil

# ── Save JSON results
with open(os.path.join(SAVE_DIR, 'all_run_records.json'), 'w') as f:
    json.dump(all_records, f, indent=2)

with open(os.path.join(SAVE_DIR, 'final_results.json'), 'w') as f:
    json.dump(final_results, f, indent=2)

print(f'Results saved to {SAVE_DIR}')

# ── Zip and download
zip_path = '/content/scribble_runs_backup'
shutil.make_archive(zip_path, 'zip', SAVE_DIR)
print(f'Archive: {zip_path}.zip')

from google.colab import files
files.download(f'{zip_path}.zip')